# 08 — All Models Combined (Final 6-Model Ensemble)

Loads all 6 predictors (3 standalone DL models + 3 classical sub-ensembles), collects their softmax probability vectors for every test image, and produces a final prediction via confidence-weighted (summed) voting.

**Run notebooks 01–06 first** to ensure all model files exist in `saved_models/`.

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
import joblib
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME    = "08_AllModelsCombined_FinalEnsemble"

DATASET_PATH     = "../MRI_DATASET/"
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

BATCH_SIZE       = 32
RANDOM_SEED      = 42

SAVED_MODELS_DIR = "../saved_models/"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Classes : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

Three input sizes are needed: CNN uses 224×224; InceptionV3 and Xception use 299×299.

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

test_gen_224 = datagen.flow_from_directory(
    TEST_DIR, target_size=(224, 224), batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False
)
test_gen_299_inc = datagen.flow_from_directory(
    TEST_DIR, target_size=(299, 299), batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False
)
test_gen_299_xcp = datagen.flow_from_directory(
    TEST_DIR, target_size=(299, 299), batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False
)

y_true = test_gen_224.classes

print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices : {test_gen_224.class_indices}")
print(f"Test samples  : {test_gen_224.samples}")
print("  CNN generator    : 224×224")
print("  InceptionV3 gen  : 299×299")
print("  Xception gen     : 299×299")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Preprocessing handled by ImageDataGenerator (rescale 1/255) for each generator.
print("✓ All generators use rescale=1/255 — no further preprocessing needed")

## Section 5: Model Loading

In [ ]:
print("Loading standalone DL models...")
cnn_model = load_model(SAVED_MODELS_DIR + 'cnn_model.h5')
inc_model = load_model(SAVED_MODELS_DIR + 'inceptionv3_model.h5')
xcp_model = load_model(SAVED_MODELS_DIR + 'xception_model.h5')
print("✓ CNN, InceptionV3, Xception loaded")

print("\nBuilding feature extractors (Dense 'feature_layer' → 256 outputs)...")
cnn_feat_ext = Model(inputs=cnn_model.input,
                     outputs=cnn_model.get_layer('feature_layer').output,
                     name='cnn_feat')
inc_feat_ext = Model(inputs=inc_model.input,
                     outputs=inc_model.get_layer('feature_layer').output,
                     name='inc_feat')
xcp_feat_ext = Model(inputs=xcp_model.input,
                     outputs=xcp_model.get_layer('feature_layer').output,
                     name='xcp_feat')
print("✓ Feature extractors ready")

print("\nLoading classical ensemble models...")
cnn_ensemble = joblib.load(SAVED_MODELS_DIR + 'cnn_ensemble_model.pkl')
inc_ensemble = joblib.load(SAVED_MODELS_DIR + 'inceptionv3_ensemble_model.pkl')
xcp_ensemble = joblib.load(SAVED_MODELS_DIR + 'xception_ensemble_model.pkl')
print("✓ Classical ensemble models loaded")

print("\n✓ All 6 predictors ready for inference")

## Section 6: Feature Extraction & Prediction Pipeline

In [ ]:
# ─── Softmax probability vectors from the 3 standalone DL models ─────────────
print("CNN predictions...")
test_gen_224.reset()
prob_cnn = cnn_model.predict(test_gen_224, verbose=1)         # (N, 4)

print("InceptionV3 predictions...")
test_gen_299_inc.reset()
prob_inc = inc_model.predict(test_gen_299_inc, verbose=1)     # (N, 4)

print("Xception predictions...")
test_gen_299_xcp.reset()
prob_xcp = xcp_model.predict(test_gen_299_xcp, verbose=1)    # (N, 4)
print("✓ DL model probabilities obtained")

# ─── Feature extraction → classical ensemble probability vectors ──────────────
print("\nCNN features → classical ensemble...")
test_gen_224.reset()
feat_cnn     = cnn_feat_ext.predict(test_gen_224, verbose=1)
prob_cnn_ens = cnn_ensemble.predict_proba(feat_cnn)

print("InceptionV3 features → classical ensemble...")
test_gen_299_inc.reset()
feat_inc     = inc_feat_ext.predict(test_gen_299_inc, verbose=1)
prob_inc_ens = inc_ensemble.predict_proba(feat_inc)

print("Xception features → classical ensemble...")
test_gen_299_xcp.reset()
feat_xcp     = xcp_feat_ext.predict(test_gen_299_xcp, verbose=1)
prob_xcp_ens = xcp_ensemble.predict_proba(feat_xcp)
print("✓ Classical ensemble probabilities obtained")

# ─── Confidence-weighted final vote: sum of 6 probability vectors ────────────
combined_probs = (prob_cnn + prob_inc + prob_xcp +
                  prob_cnn_ens + prob_inc_ens + prob_xcp_ens)
final_proba    = combined_probs / combined_probs.sum(axis=1, keepdims=True)
y_pred_final   = np.argmax(combined_probs, axis=1)
confidence     = np.max(final_proba, axis=1)

print(f"\n✓ Final predictions computed for {len(y_pred_final)} images")
print(f"  Mean confidence : {confidence.mean():.4f}")
print(f"  Min confidence  : {confidence.min():.4f}")

## Section 7: Model Evaluation

In [ ]:
# Per-model accuracy
acc_cnn     = accuracy_score(y_true, np.argmax(prob_cnn, axis=1))
acc_inc     = accuracy_score(y_true, np.argmax(prob_inc, axis=1))
acc_xcp     = accuracy_score(y_true, np.argmax(prob_xcp, axis=1))
acc_cnn_ens = accuracy_score(y_true, cnn_ensemble.predict(feat_cnn))
acc_inc_ens = accuracy_score(y_true, inc_ensemble.predict(feat_inc))
acc_xcp_ens = accuracy_score(y_true, xcp_ensemble.predict(feat_xcp))
acc_final   = accuracy_score(y_true, y_pred_final)

# Bar chart comparison
model_labels = ['01 CNN', '03 InceptionV3', '05 Xception',
                '02 CNN+Ens', '04 Inc+Ens', '06 Xcp+Ens', 'FINAL']
accuracies   = [acc_cnn, acc_inc, acc_xcp, acc_cnn_ens, acc_inc_ens, acc_xcp_ens, acc_final]
colors = ['steelblue'] * 6 + ['darkgreen']
plt.figure(figsize=(11, 6))
bars = plt.bar(model_labels, accuracies, color=colors, edgecolor='black')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{acc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.ylim(0, 1.12)
plt.ylabel('Test Accuracy')
plt.title('All Models — Accuracy Comparison (FINAL = 6-model ensemble)')
plt.xticks(rotation=15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Final ensemble confusion matrix
cm = confusion_matrix(y_true, y_pred_final)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Final 6-Model Ensemble — Confusion Matrix (Acc: {acc_final:.4f})')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

print("\nFinal Ensemble — Classification Report")
print("=" * 60)
print(classification_report(y_true, y_pred_final, target_names=CLASS_NAMES))

# ROC curves
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
plt.figure(figsize=(8, 6))
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], final_proba[:, i])
    plt.plot(fpr, tpr, label=f'{cls} (AUC = {auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Final Ensemble — ROC Curve'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

# PR curves
plt.figure(figsize=(8, 6))
for i, cls in enumerate(CLASS_NAMES):
    p, r, _ = precision_recall_curve(y_true_bin[:, i], final_proba[:, i])
    ap = average_precision_score(y_true_bin[:, i], final_proba[:, i])
    plt.plot(r, p, label=f'{cls} (AP = {ap:.2f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Final Ensemble — Precision-Recall Curve'); plt.legend(loc='upper right')
plt.tight_layout(); plt.show()

## Section 8: Save Model

In [ ]:
save_path = SAVED_MODELS_DIR + 'final_ensemble_predictions.pkl'
with open(save_path, 'wb') as f:
    pickle.dump({
        'y_true': y_true,
        'y_pred': y_pred_final,
        'combined_probs': combined_probs,
        'final_proba': final_proba,
        'class_names': CLASS_NAMES
    }, f)
print(f"✓ Final predictions saved to: {save_path}")
print("  To reproduce: load all 6 sub-models and run Sections 5–6.")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Test samples : {len(y_true)}")
print(f"Classes      : {CLASS_NAMES}")
print(f"Seed         : {RANDOM_SEED}")
print("-" * 60)
print("INDIVIDUAL MODEL ACCURACIES:")
print(f"  01 CNN standalone         : {acc_cnn:.4f}")
print(f"  03 InceptionV3 standalone : {acc_inc:.4f}")
print(f"  05 Xception standalone    : {acc_xcp:.4f}")
print(f"  02 CNN + Ensemble         : {acc_cnn_ens:.4f}")
print(f"  04 InceptionV3 + Ensemble : {acc_inc_ens:.4f}")
print(f"  06 Xception + Ensemble    : {acc_xcp_ens:.4f}")
print("-" * 60)
print(f"  FINAL ENSEMBLE (6 models) : {acc_final:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)